# テキストベクトル化手法の比較実験（train_baseline 複製）

- **目的**: テキスト→ベクトルの手法を**網羅的に**試し、sentence-transformers のありがたみがわかるようにする。
- **実装**: ベクトル化ロジックは **`text_vectors.py`** に集約。ノートは build_X から呼ぶだけ。
- **ベース**: 34 特徴（3C3 含む）。各実験は「ベース ＋ movie_info のベクトル化 1 種類」のみ。
- **検証**: 時系列 CV（2013〜2016 を val）。ベクトル化は **各 fold の tr のみ** で fit。
- **レパートリー**: TF-IDF(50/100/svd20), Count(50/svd20), Hash(50/svd20), LDA(10/20), NMF(10/20), Doc2Vec, Word2Vec 平均, sentence-transformers（要事前計算）。

In [15]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

import sys, os
if os.path.basename(os.getcwd()) == "終わった実験環境":
    sys.path.insert(0, os.path.abspath(".."))
from preprocess import load_train_test
from feature_engineering import create_features
from lib.text_vectors import build_vectors, get_available_configs, get_config_descriptions

try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False

print("Setup complete. sentence_transformers:", HAS_SENTENCE_TRANSFORMERS)

Setup complete. sentence_transformers: True


In [16]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
seed_everything(42)

In [17]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [18]:
train = create_features(train)
test = create_features(test)
for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype("category")
        test[col] = test[col].fillna("missing").astype("category")
for col in ["runtime"]:
    if col in train.columns and col in test.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)
print("特徴量作成完了.")

特徴量作成完了.


In [19]:
# 3C3（train_baseline と同じ）
def _ts_te_col(df_tr, df_te, col, target_name="target", m=10):
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, te_arr

for col in ["critic_name", "production_company"]:
    if col not in train.columns: continue
    st, ta = _ts_te_col(train, test, col, "target", m=10)
    train[f"{col}_te_ts"] = st.reindex(train.index).fillna(train["target"].mean())
    test[f"{col}_te_ts"] = ta
for c in ["critic_name_te_ts", "production_company_te_ts"]:
    if c not in train.columns: continue
    train[c + "_bin"] = pd.cut(train[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
    test[c + "_bin"] = pd.cut(test[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
train["runtime_bin"] = pd.cut(train["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
test["runtime_bin"] = pd.cut(test["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
def _movie_age_bin(x):
    if pd.isna(x) or x < 0: return 0
    if x < 365: return 1
    if x < 365 * 5: return 2
    if x < 365 * 20: return 3
    return 4
train["movie_age_bin"] = train["movie_age_days"].apply(_movie_age_bin)
test["movie_age_bin"] = test["movie_age_days"].apply(_movie_age_bin)
train["release_decade"] = (train["release_year"] // 10 * 10).fillna(1990)
test["release_decade"] = (test["release_year"] // 10 * 10).fillna(1990)
print("3C3 特徴量追加完了.")

3C3 特徴量追加完了.


In [ ]:
# ベース 34 特徴。ベクトル化は text_vectors.build_vectors で tr のみ fit。
FEATURES_BASE = [
    "rotten_tomatoes_link", "critic_name", "top_critic", "publisher_name",
    "movie_title", "movie_info", "content_rating", "directors", "authors", "actors",
    "runtime", "production_company",
    "review_year", "review_month", "review_dayofweek",
    "release_year", "release_month", "release_dayofweek", "movie_age_days",
    "genre_Drama", "genre_Comedy", "genre_Action", "genre_Mystery",
    "genre_Fantasy", "genre_Romance", "genre_Horror", "genre_Documentary",
    "critic_name_te_ts", "production_company_te_ts",
    "critic_name_te_ts_bin", "production_company_te_ts_bin",
    "runtime_bin", "movie_age_bin", "release_decade",
]

def _mi_series(df):
    return df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x))

def _concat_mi_title(df):
    """movie_title と movie_info を結合した 1 文書（タイトル＋あらすじ）。"""
    t = df["movie_title"].apply(lambda x: "" if pd.isna(x) else str(x))
    m = df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x))
    return t + " . " + m

def build_X_for_experiment(config_name, train_df, test_df, tr_idx, val_idx, y_train, st_emb_train=None, st_emb_test=None):
    tr_df = train_df.iloc[tr_idx].copy()
    val_df = train_df.iloc[val_idx].copy() if val_idx is not None and len(val_idx) else pd.DataFrame()
    te_df = test_df.copy() if test_df is not None else None
    tr_df["target"] = y_train[tr_idx]
    base = [c for c in FEATURES_BASE if c in tr_df.columns]

    if config_name == "base":
        return tr_df[base].copy(), val_df[base].copy() if len(val_df) else None, te_df[base].copy() if te_df is not None else None, base

    mi_tr = _mi_series(tr_df)
    mi_val = _mi_series(val_df) if len(val_df) else None
    mi_te = _mi_series(te_df) if te_df is not None and len(te_df) else None

    # 結合テキストでベクトル化する config はタイトル＋あらすじを渡す
    if config_name.startswith("concat_mi_title_"):
        txt_tr = _concat_mi_title(tr_df)
        txt_val = _concat_mi_title(val_df) if len(val_df) else None
        txt_te = _concat_mi_title(te_df) if te_df is not None and len(te_df) else None
        out = build_vectors(config_name, txt_tr, txt_val, txt_te)
    elif config_name == "sentence_transformer_32" and st_emb_train is not None and st_emb_test is not None:
        st_emb_val = st_emb_train[val_idx] if val_idx is not None and len(val_idx) else None
        out = build_vectors(config_name, mi_tr, mi_val, mi_te, st_emb_tr=st_emb_train[tr_idx], st_emb_val=st_emb_val, st_emb_te=st_emb_test)
    else:
        out = build_vectors(config_name, mi_tr, mi_val, mi_te)

    tr_mat, val_mat, te_mat, prefix = out
    if tr_mat is None or prefix is None:
        return tr_df[base].copy(), val_df[base].copy() if len(val_df) else None, te_df[base].copy() if te_df is not None else None, base

    n_col = tr_mat.shape[1]
    for i in range(n_col):
        tr_df[f"{prefix}_{i}"] = tr_mat[:, i]
        if val_mat is not None: val_df[f"{prefix}_{i}"] = val_mat[:, i]
        if te_mat is not None: te_df[f"{prefix}_{i}"] = te_mat[:, i]
    extra = [f"{prefix}_{i}" for i in range(n_col)]
    feats = base + extra
    return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

# 実験設定: text_vectors のレパートリーから取得。sentence_transformer は下のセル実行で追加可。
EXPERIMENT_CONFIGS = get_available_configs(include_sentence_transformer=False)
print("実験設定:", len(EXPERIMENT_CONFIGS), "件", EXPERIMENT_CONFIGS)

実験設定: 15 件 ['base', 'tfidf_50', 'tfidf_100', 'tfidf_svd20', 'count_50', 'count_svd20', 'hash_50', 'hash_svd20', 'lda_10', 'lda_20', 'nmf_10', 'nmf_20', 'concat_mi_title_tfidf_50', 'concat_mi_title_tfidf_svd20', 'concat_mi_title_count_50']


### 各手法の意味（text_vectors.py で定義）

| config | 説明 |
|--------|------|
| base | 34 特徴のみ |
| tfidf_50 / tfidf_100 | TF-IDF 50 or 100 次元 |
| tfidf_svd20 | TF-IDF(200) → SVD 20 次元 |
| count_50 | CountVectorizer 50 次元（BoW） |
| count_svd20 | Count(200) → SVD 20 次元 |
| hash_50 | HashingVectorizer 先頭 50 次元 |
| hash_svd20 | Hash(512) → SVD 20 次元 |
| lda_10 / lda_20 | LDA 10 or 20 トピック |
| nmf_10 / nmf_20 | NMF 10 or 20 次元 |
| doc2vec_32 | Doc2Vec 32 次元（gensim 要） |
| word2vec_32 | Word2Vec 平均 32 次元（gensim 要） |
| sentence_transformer_32 | sentence-transformers 32 次元（下のセルで事前計算） |
| concat_mi_title_tfidf_50 | **タイトル＋あらすじ結合** → TF-IDF 50 次元 |
| concat_mi_title_tfidf_svd20 | タイトル＋あらすじ結合 → TF-IDF→SVD 20 次元 |
| concat_mi_title_count_50 | タイトル＋あらすじ結合 → Count 50 次元 |

### sentence-transformers 埋め込みの事前計算（オプション）

ライブラリが入っていれば実行。実行後は `EXPERIMENT_CONFIGS` に `sentence_transformer_32` を手動で追加して CV を回す。

In [14]:
# sentence_transformers で movie_info を固定次元ベクトルに。全データ一括でOK（target を使わないため）。
ST_EMB_TRAIN = None
ST_EMB_TEST = None
if HAS_SENTENCE_TRANSFORMERS:
    try:
        model = SentenceTransformer("all-MiniLM-L6-v2")  # 384次元 → 先頭32だけ使う
        mi_train = train["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x))
        mi_test = test["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x))
        ST_EMB_TRAIN = model.encode(mi_train.tolist(), show_progress_bar=True)
        ST_EMB_TEST = model.encode(mi_test.tolist(), show_progress_bar=True)
        # 32次元に切り詰め
        ST_EMB_TRAIN = ST_EMB_TRAIN[:, :32]
        ST_EMB_TEST = ST_EMB_TEST[:, :32]
        print(f"sentence_transformers 埋め込み: train {ST_EMB_TRAIN.shape}, test {ST_EMB_TEST.shape}")
        if "sentence_transformer_32" not in EXPERIMENT_CONFIGS:
            EXPERIMENT_CONFIGS.append("sentence_transformer_32")
    except Exception as e:
        print("sentence_transformers エラー:", e)
else:
    print("sentence_transformers 未インストール。pip install sentence-transformers で追加可能。")

Batches:   0%|          | 0/20423 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))
print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])


In [12]:
# 再開用（任意）: docs/01_PREPROCESS.md §6 参照。途中で止めた場合はここを有効にして下の CV を実行
RECORDED_PARTIAL_RESULTS = [
    {"config": "base", "n_feat": 34, "CV_AUC": 0.7600},
    {"config": "tfidf_50", "n_feat": 84, "CV_AUC": 0.7600},
    {"config": "tfidf_100", "n_feat": 134, "CV_AUC": 0.7600},
    {"config": "tfidf_svd20", "n_feat": 54, "CV_AUC": 0.7601},
    {"config": "count_50", "n_feat": 84, "CV_AUC": 0.7603},
    {"config": "count_svd20", "n_feat": 54, "CV_AUC": 0.7603},
    {"config": "hash_50", "n_feat": 84, "CV_AUC": 0.7600},
    {"config": "hash_svd20", "n_feat": 54, "CV_AUC": 0.7601},
    {"config": "lda_10", "n_feat": 44, "CV_AUC": 0.7606},
    {"config": "lda_20", "n_feat": 54, "CV_AUC": 0.7603},
    {"config": "nmf_10", "n_feat": 44, "CV_AUC": 0.7600},
]
START_FROM_CONFIG = "nmf_20"  # ここから実行。全件やり直す場合は RECORDED_PARTIAL_RESULTS = [] と START_FROM_CONFIG = None

In [ ]:
y = train["target"].values
lgb_params = {"objective": "binary", "metric": "auc", "boosting_type": "gbdt", "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31, "random_state": 42, "verbosity": -1}
# 再開時: 上のセルで RECORDED_PARTIAL_RESULTS と START_FROM_CONFIG を設定済みならそれを使用（未定義なら全件）
try:
    _partial = RECORDED_PARTIAL_RESULTS
    _start = START_FROM_CONFIG
except NameError:
    _partial = []
    _start = None
results = list(_partial) if _partial else []
configs_to_run = EXPERIMENT_CONFIGS if _start is None else EXPERIMENT_CONFIGS[EXPERIMENT_CONFIGS.index(_start):]
for config_name in configs_to_run:
    print(f"  {config_name}...", end=" ")
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(time_splits):
        X_tr, X_val, _, feats = build_X_for_experiment(config_name, train, test, tr_idx, val_idx, y, st_emb_train=ST_EMB_TRAIN, st_emb_test=ST_EMB_TEST)
        if X_val is None or len(X_val) == 0: continue
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
        auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
        fold_scores.append(auc)
    mean_auc = np.mean(fold_scores) if fold_scores else np.nan
    n_feat = len(feats)
    results.append({"config": config_name, "n_feat": n_feat, "CV_AUC": mean_auc})
    print(f"n_feat={n_feat}, AUC={mean_auc:.4f}")
res_df = pd.DataFrame(results)
base_auc = res_df.loc[res_df["config"] == "base", "CV_AUC"].iloc[0]
res_df["delta_from_base"] = res_df["CV_AUC"] - base_auc
res_df = res_df.sort_values("CV_AUC", ascending=False)
print(res_df.to_string(index=False))
display(res_df)

  nmf_20... 

NameError: name 'ST_EMB_TRAIN' is not defined

### 提出用: CV 最良 config で全 train 学習 → submission.csv

In [15]:
# CV で一番良かった config（base 除く）で全データ学習し、test を予測。res_df が無い場合はデフォルト config を使用（単体実行可）
try:
    best_config = res_df.loc[res_df["config"] != "base", "config"].iloc[0]
except NameError:
    best_config = "lda_10"  # 単体実行時用デフォルト（CV セルを未実行の場合）
print(f"Best config (excl. base): {best_config}")
tr_all = np.arange(len(train))
X_tr, _, X_te, feats = build_X_for_experiment(best_config, train, test, tr_all, None, y, st_emb_train=ST_EMB_TRAIN, st_emb_test=ST_EMB_TEST)
model_full = lgb.LGBMClassifier(**lgb_params).fit(X_tr, y)
pred = model_full.predict_proba(X_te)[:, 1]
submission = pd.DataFrame({"ID": test["ID"], "target": pred})
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv (rows: {len(submission):,}) [config={best_config}]")

Best config (excl. base): lda_10
Saved submission.csv (rows: 40,716) [config=lda_10]
